# Visualizing regression results

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../../data/ckpts/rosen'

PARAMS_PATH = '../../3-REGRESSION-MODELS/reports/complete-parameter-estimates.csv'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH)
len(fs)

26

## Importing parameters

In [5]:
dfps = pd.read_csv(PARAMS_PATH)
dfps.head()

,param,beta,std,k,se
0,nx,0.141443,0.017831,1290,0.000496
1,ny,-0.021446,0.007478,1290,0.000208
2,self,-0.086012,0.421904,1290,0.011747


## Processing individual datasets

In [6]:
COEF_VAR = 'I'

In [7]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [8]:
VAL_COL = 'resid'
group_by_cols = ['lang', 'turn_delta', 'self']

In [9]:
df_scale, df_regression = [], []

In [10]:
for f in tqdm(fs):
    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id', # 'AVAR',
        'nx', 'ny']).to_pandas()

    df = df.loc[
        (df['nx'] >= 5)
        & (df['ny'] >= 5)
    ]

    if ('CANDOR' in f.split('/')[-1]):
        df['corpus'] = 'CANDOR'

    if ('MICASE' in f.split('/')[-1]):
        df['corpus'] = 'MICASE'

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)

    df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])

    df['lang'] = [c.split('-')[1] if (c != 'yiddish') else 'yid' for c in df['corpus']]

    # if ('CANDOR' in f.split('/')[-1]) or ('grace' in f.split('/')[-1]):
    #     df['turn_delta'].loc[df['self'] == 0] -= 1

    if VAL_COL == 'resid':
        df['pred'] = (df['nx'] * dfps['beta'].loc[dfps['param'].isin(['nx'])].values[0]) + (df['ny'] * dfps['beta'].loc[dfps['param'].isin(['ny'])].values[0])

        df['resid'] = df[COEF_VAR] - df['pred']

    df_regression += [
        df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('mean').reset_index()
    ]

    df_regression[-1]['std'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('std').reset_index()[VAL_COL]

    df_regression[-1]['k'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('count').reset_index()[VAL_COL]

    df_regression[-1]['se'] = df_regression[-1]['std'] / np.sqrt(df_regression[-1]['k'])

df_regression = pd.concat(df_regression, ignore_index=True)

  0%|          | 0/26 [00:00<?, ?it/s]

In [11]:
df0 = df_regression[['turn_delta', 'self'] + [VAL_COL]].groupby(by=['turn_delta', 'self']).agg('mean').reset_index()
df0.head()

,turn_delta,self,resid
0,1,0,-0.202470
1,1,1,-0.383084
2,2,0,-0.203236
3,2,1,-0.366307
4,3,0,-0.197157


In [12]:
df1 = df_regression[['turn_delta', 'self', 'lang'] + [VAL_COL]].groupby(by=['turn_delta', 'self', 'lang']).agg('mean').reset_index()
df1.head(n=25)

,turn_delta,self,lang,resid
0,1,0,cro,-0.168931
1,1,0,deu,-0.132395
2,1,0,eng,-0.087474
3,1,0,fin,0.028941
4,1,0,fra,-0.123953
5,1,0,ko,-0.146623
6,1,0,spa,-0.150364
7,1,0,yid,-0.765728
8,1,0,zho,-0.667518
9,1,1,cro,-0.202558


## Plotly vis

In [13]:
import plotly.graph_objects as go

### Aggregate/total visualization

Some of the corpora have different structures overall. This means that, for say CANDOR and the MIAO corpora, there is a strange distribution of turns such that self is always even turns away and other is always odd turns away. Especially because CANDOR is such a large component of the data, this caused a bumpier distribution than one would have expected.

To solve for this, we averaged odd versus even turns, smoothening out the overall visualization of the distribution.

In [14]:
sel_1 = (df0['turn_delta'] <= 200) & (df0['turn_delta'] > 0) & ((df0['turn_delta'] % 2) != 0)

In [15]:
fig = go.Figure()

In [16]:
sel = df0['self'] == 1
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange'
        )
    )
)

sel = df0['self'] == 0
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue'
        )
    )
)

fig.update_layout(template='xgridoff')

In [17]:
fig.write_html(os.path.join(OUTPUTS_PATH, 'all-corpora.html'))
fig.write_image(os.path.join(OUTPUTS_PATH, 'all-corpora.png'), scale=5)

### Per corpus vis

In [18]:
figs = []

In [19]:
for corpus in df1.lang.unique():
    sub = df1.loc[df1['lang'].isin([corpus])]

    fig = go.Figure()

    sel_t = (sub['turn_delta'] > 0) & (sub['turn_delta'] <= 200)

    sel = sub['self'] == 1
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='self',
            showlegend=True,
            legendgroup='self',
            line = dict(
                color='orange'
            )
        )
    )

    sel = sub['self'] == 0
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='other',
            showlegend=True,
            legendgroup='other',
            line = dict(
                color='royalblue'
            )
        )
    )

    fig.update_layout(template='xgridoff')

    figs += [fig]

In [20]:
for corpus, plot in zip(df1.lang.unique(), figs):
    print(corpus)
    plot.show()
    plot.write_html(os.path.join(OUTPUTS_PATH, corpus+'.html'))
    plot.write_image(os.path.join(OUTPUTS_PATH, corpus+'.png'), scale=5)

cro


deu


eng


fin


fra


ko


spa


yid


zho
